
# 人材定着予測

**Public スコア 0.5568419345849898**


In [1]:

# ============================================================
# ① import
# ============================================================
import os
import re
import sys
import subprocess
import warnings
import datetime as dt
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd

try:
    from catboost import CatBoostClassifier
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "catboost"])
    from catboost import CatBoostClassifier

from sklearn.metrics import log_loss, roc_auc_score
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 120)

ID_COLUMN = "社員ID"
TARGET_COLUMN = "10年定着ラベル"
TEXT_COLUMNS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]
JST = ZoneInfo("Asia/Tokyo")


In [2]:

# ============================================================
# ② Google Drive をマウント
# ============================================================
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Google Drive mount skipped:", e)


Mounted at /content/drive


In [3]:

# ============================================================
# ③ パス設定
# ============================================================
MODEL_NAME = "Sample0812"

FILES = {
    "persona_train": "employee_persona_train.csv",
    "persona_test": "employee_persona_test.csv",
    "monthly_train": "employee_monthly_train.csv",
    "monthly_test": "employee_monthly_test.csv",
    "monthly_train_full": "employee_monthly_train_full.csv",
    "sample_submission": "sample_submission.csv",
}

DRIVE_DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/data/"
LOCAL_DATA_DIR = "/content"
CHATGPT_DATA_DIR = "/mnt/data"

OUT_DIR = "/content/drive/MyDrive/Colab Notebooks/output/"
if not os.path.exists("/content/drive/MyDrive"):
    OUT_DIR = "/mnt/data/output/"
os.makedirs(OUT_DIR, exist_ok=True)

def resolve_path(filename: str) -> str:
    candidates = [
        os.path.join(DRIVE_DATA_DIR, filename),
        os.path.join(LOCAL_DATA_DIR, filename),
        os.path.join(CHATGPT_DATA_DIR, filename),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(
        f"Not found: {filename}\nTried:\n- " + "\n- ".join(candidates)
    )

PATHS = {key: resolve_path(filename) for key, filename in FILES.items()}

for key, path in PATHS.items():
    print(f"{key:20s}: {path}")
print("OUT_DIR             :", OUT_DIR)


persona_train       : /content/drive/MyDrive/Colab Notebooks/data/employee_persona_train.csv
persona_test        : /content/drive/MyDrive/Colab Notebooks/data/employee_persona_test.csv
monthly_train       : /content/drive/MyDrive/Colab Notebooks/data/employee_monthly_train.csv
monthly_test        : /content/drive/MyDrive/Colab Notebooks/data/employee_monthly_test.csv
monthly_train_full  : /content/drive/MyDrive/Colab Notebooks/data/employee_monthly_train_full.csv
sample_submission   : /content/drive/MyDrive/Colab Notebooks/data/sample_submission.csv
OUT_DIR             : /content/drive/MyDrive/Colab Notebooks/output/


In [4]:

# ============================================================
# ④ 前処理
# ============================================================

NUMERIC_COLS = [
    "月例給与_円",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "担当プロジェクト数", "顧客満足度評価",
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
]

TREND_COLS = [
    "月例給与_円", "残業時間", "研修時間", "情報共有件数", "在宅勤務日数",
    "360度評価_主体度", "360度評価_信頼度", "360度評価_共有貢献度",
]

DYNAMIC_COLS = ["部署ID", "職種", "役割", "等級", "上司ID", "勤務地"]

RECENT_COLS = [
    "残業時間", "情報共有件数", "研修時間", "欠勤日数", "有給取得日数",
    "在宅勤務日数", "上司との面談実施回数", "月例給与_円",
    "360度評価_主体度", "360度評価_信頼度", "360度評価_共有貢献度",
    "顧客満足度評価",
]

INTERACTION_PAIRS = [
    ("入社区分", "採用経路"),
    ("入社区分", "初期職種"),
    ("採用経路", "初期職種"),
    ("専攻分野", "初期職種"),
    ("前職職種", "初期職種"),
    ("初期部署ID", "初期職種"),
    ("初期等級", "初期役割"),
    ("初期勤務地", "初期職種"),
    ("性別", "初期職種"),
    ("初期職種", "職種__last"),
    ("初期部署ID", "部署ID__last"),
    ("初期等級", "等級__last"),
    ("初期役割", "役割__last"),
    ("職種__first", "職種__last"),
    ("部署ID__first", "部署ID__last"),
    ("上司ID__first", "上司ID__last"),
]


def load_raw(paths: dict):
    persona_train = pd.read_csv(paths["persona_train"])
    persona_test = pd.read_csv(paths["persona_test"])
    monthly_train = pd.read_csv(paths["monthly_train"])
    monthly_test = pd.read_csv(paths["monthly_test"])
    monthly_train_full = pd.read_csv(paths["monthly_train_full"])

    sample_submission = pd.read_csv(
        paths["sample_submission"],
        header=None,
        names=[ID_COLUMN, "10年定着確率"],
    )
    return (
        persona_train,
        persona_test,
        monthly_train,
        monthly_test,
        monthly_train_full,
        sample_submission,
    )


def audit_raw_data(persona_train, persona_test, monthly_train, monthly_test, monthly_train_full):
    assert TARGET_COLUMN in persona_train.columns
    assert TARGET_COLUMN not in persona_test.columns
    assert persona_train[ID_COLUMN].is_unique
    assert persona_test[ID_COLUMN].is_unique
    assert monthly_train["経過月数"].min() >= 0
    assert monthly_train["経過月数"].max() <= 23
    assert monthly_test["経過月数"].min() >= 0
    assert monthly_test["経過月数"].max() <= 23

    print("persona_train      :", persona_train.shape)
    print("persona_test       :", persona_test.shape)
    print("monthly_train      :", monthly_train.shape,
          "months", monthly_train["経過月数"].min(), "〜", monthly_train["経過月数"].max())
    print("monthly_test       :", monthly_test.shape,
          "months", monthly_test["経過月数"].min(), "〜", monthly_test["経過月数"].max())
    print("monthly_train_full :", monthly_train_full.shape,
          "months", monthly_train_full["経過月数"].min(), "〜", monthly_train_full["経過月数"].max())
    print("※ train_full の24〜119か月は特徴量に使用しません。")


def remove_early_leavers(persona_train, monthly_train):
    early_ids = monthly_train.loc[
        monthly_train["月末在籍状態"].eq("退職"), ID_COLUMN
    ].unique()

    persona_use = persona_train.loc[
        ~persona_train[ID_COLUMN].isin(early_ids)
    ].copy()

    monthly_use = monthly_train.loc[
        ~monthly_train[ID_COLUMN].isin(early_ids)
    ].copy()

    return (
        persona_use.reset_index(drop=True),
        monthly_use.reset_index(drop=True),
        early_ids,
    )


def make_learning_features(monthly: pd.DataFrame) -> pd.DataFrame:

    z = monthly[[ID_COLUMN, "経過月数", "自己学習（詳細）"]].copy()
    z = z[
        z["自己学習（詳細）"].notna()
        & z["自己学習（詳細）"].ne("受講なし")
    ]

    out = pd.DataFrame(index=monthly[ID_COLUMN].drop_duplicates().values)
    out.index.name = ID_COLUMN

    out["学習_active_months"] = z.groupby(ID_COLUMN)["経過月数"].nunique()

    tokens = z.assign(
        token=z["自己学習（詳細）"].astype(str).str.split("｜")
    ).explode("token")

    extracted = tokens["token"].str.extract(r"^(.*)：([0-9.]+)時間$")
    tokens["course"] = extracted[0]
    tokens["hours"] = pd.to_numeric(extracted[1], errors="coerce")
    tokens = tokens.dropna(subset=["course", "hours"])

    out["学習_course_events"] = tokens.groupby(ID_COLUMN).size()
    out["学習_unique_courses"] = tokens.groupby(ID_COLUMN)["course"].nunique()
    out["学習_total_hours_parsed"] = tokens.groupby(ID_COLUMN)["hours"].sum()

    course_buckets = {
        "DX_IT": ["Python", "SQL", "クラウド", "データ", "システム", "RPA", "AI", "統計", "アジャイル"],
        "対人営業": ["顧客", "提案", "交渉", "CRM", "アカウント", "ファシリ", "コミュニケーション"],
        "管理リスク": ["リスク", "コンプライアンス", "内部統制", "労務", "管理会計", "品質管理"],
        "組織業務": ["組織", "人材", "業務", "プロセス", "プロジェクト"],
    }

    course_text = tokens["course"].astype(str)
    for bucket_name, words in course_buckets.items():
        mask = course_text.apply(lambda s: any(word in s for word in words))
        out[f"学習時間_{bucket_name}"] = (
            tokens.loc[mask].groupby(ID_COLUMN)["hours"].sum()
        )

    return out.fillna(0.0)


def make_employee_features(persona: pd.DataFrame, monthly: pd.DataFrame) -> pd.DataFrame:
    assert monthly["経過月数"].min() >= 0
    assert monthly["経過月数"].max() <= 23, (
        "LEAK BLOCKED: make_employee_features() に24か月以降のデータが渡されています。"
    )

    base = persona.copy()
    base["入社日_dt"] = pd.to_datetime(base["入社日"])
    base["入社月"] = base["入社日_dt"].dt.month.astype(str)
    base["入社四半期"] = base["入社日_dt"].dt.quarter.astype(str)

    for col in TEXT_COLUMNS:
        base[f"{col}__文字数"] = base[col].fillna("").astype(str).str.len()

    base["前職と初期職種一致"] = (
        base["前職職種"].fillna("__NA__").astype(str)
        == base["初期職種"].fillna("__NA__").astype(str)
    ).astype("int8")

    m = monthly.sort_values([ID_COLUMN, "経過月数"]).copy()
    g = m.groupby(ID_COLUMN, sort=False)
    blocks = []

    for stat in ["mean", "std", "min", "max", "first", "last"]:
        tmp = getattr(g[NUMERIC_COLS], stat)()
        tmp.columns = [f"{c}__{stat}" for c in tmp.columns]
        blocks.append(tmp)

    early6 = (
        m.loc[m["経過月数"].between(0, 5)]
        .groupby(ID_COLUMN)[TREND_COLS].mean()
    )
    late6 = (
        m.loc[m["経過月数"].between(18, 23)]
        .groupby(ID_COLUMN)[TREND_COLS].mean()
    )
    delta = late6 - early6

    early6.columns = [f"{c}__early6" for c in early6.columns]
    late6.columns = [f"{c}__late6" for c in late6.columns]
    delta.columns = [f"{c}__delta_late_early" for c in delta.columns]
    blocks.extend([early6, late6, delta])

    dyn = pd.DataFrame(index=g.size().index)
    for col in DYNAMIC_COLS:
        dyn[f"{col}__nunique"] = g[col].nunique(dropna=True)
        dyn[f"{col}__first"] = g[col].first()
        dyn[f"{col}__last"] = g[col].last()

        z = m[[ID_COLUMN, col]].copy()
        changed = (
            z[col].ne(z.groupby(ID_COLUMN)[col].shift())
            & z.groupby(ID_COLUMN).cumcount().gt(0)
        )
        dyn[f"{col}__changes"] = changed.groupby(z[ID_COLUMN]).sum()

    dyn["休職月数"] = (
        m["月末在籍状態"].eq("休職").groupby(m[ID_COLUMN]).sum()
    )
    dyn["360度評価更新回数"] = g["360度評価更新フラグ"].sum()
    dyn["受講なし月数"] = (
        m["自己学習（詳細）"].eq("受講なし").groupby(m[ID_COLUMN]).sum()
    )
    dyn["観測月数"] = g.size()
    blocks.append(dyn)

    for n_months, start_month in [(3, 21), (6, 18), (12, 12)]:
        recent = (
            m.loc[m["経過月数"].between(start_month, 23)]
            .groupby(ID_COLUMN)[RECENT_COLS].mean()
        )
        recent.columns = [
            f"{c}__last{n_months}_mean" for c in recent.columns
        ]
        blocks.append(recent)

    blocks.append(make_learning_features(m))

    monthly_feature = pd.concat(blocks, axis=1).reset_index()
    out = base.merge(monthly_feature, on=ID_COLUMN, how="left")

    out["給与last_vs_初任給"] = (
        out["月例給与_円__last"]
        / out["初任給_円"].replace(0, np.nan)
    )
    out["給与増加率24m"] = (
        out["月例給与_円__last"]
        / out["月例給与_円__first"].replace(0, np.nan)
        - 1
    )

    for col_a, col_b in INTERACTION_PAIRS:
        if col_a in out.columns and col_b in out.columns:
            out[f"X__{col_a}__{col_b}"] = (
                out[col_a].fillna("__NA__").astype(str)
                + "|"
                + out[col_b].fillna("__NA__").astype(str)
            )

    return out


def prepare_catboost_matrix(train_feature, test_feature):
    drop_cols = [
        ID_COLUMN, TARGET_COLUMN, "入社日", "入社日_dt",
        *TEXT_COLUMNS,
    ]

    feature_cols = [
        c for c in train_feature.columns
        if c not in drop_cols
    ]

    X_train = train_feature[feature_cols].copy()
    X_test = test_feature[feature_cols].copy()
    y = train_feature[TARGET_COLUMN].astype(int).copy()

    X_test = X_test[X_train.columns]

    cat_cols = [
        c for c in X_train.columns
        if X_train[c].dtype == "object"
    ]

    for col in cat_cols:
        X_train[col] = X_train[col].fillna("__MISSING__").astype(str)
        X_test[col] = X_test[col].fillna("__MISSING__").astype(str)

    return X_train, y, X_test, cat_cols, feature_cols


def make_time_masks(train_feature):
    dates = pd.to_datetime(train_feature["入社日"])

    main_cutoff = (
        dates.max().to_period("M") - 11
    ).to_timestamp()
    prior_cutoff = main_cutoff - pd.DateOffset(years=1)

    cal_train_mask = dates < prior_cutoff
    cal_valid_mask = (dates >= prior_cutoff) & (dates < main_cutoff)

    main_train_mask = dates < main_cutoff
    main_valid_mask = dates >= main_cutoff

    assert cal_train_mask.sum() > 0
    assert cal_valid_mask.sum() > 0
    assert main_train_mask.sum() > 0
    assert main_valid_mask.sum() > 0

    return {
        "dates": dates,
        "prior_cutoff": prior_cutoff,
        "main_cutoff": main_cutoff,
        "cal_train": cal_train_mask,
        "cal_valid": cal_valid_mask,
        "main_train": main_train_mask,
        "main_valid": main_valid_mask,
    }


def to_logit(prob):
    prob = np.clip(np.asarray(prob, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(prob / (1.0 - prob))


def fit_platt_calibrator(raw_prob, y_true):
    calibrator = LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=2000,
    )
    calibrator.fit(
        to_logit(raw_prob).reshape(-1, 1),
        np.asarray(y_true, dtype=int),
    )
    return calibrator


def apply_platt(calibrator, raw_prob):
    return calibrator.predict_proba(
        to_logit(raw_prob).reshape(-1, 1)
    )[:, 1]


def evaluate_prob(y_true, prob, name):
    ll = log_loss(y_true, prob)
    auc = roc_auc_score(y_true, prob)
    print(
        f"{name:36s} "
        f"LogLoss={ll:.6f}  AUC={auc:.6f}  "
        f"pred_mean={np.mean(prob):.5f}"
    )
    return ll, auc


def save_submission(sample_submission, test_feature, test_pred, model_name):
    pred_df = pd.DataFrame({
        ID_COLUMN: test_feature[ID_COLUMN].values,
        "10年定着確率": np.clip(test_pred, 1e-5, 1 - 1e-5),
    })

    submission = sample_submission[[ID_COLUMN]].merge(
        pred_df,
        on=ID_COLUMN,
        how="left",
        validate="one_to_one",
    )

    assert submission["10年定着確率"].notna().all()
    assert submission["10年定着確率"].between(0, 1).all()
    assert len(submission) == len(sample_submission)

    timestamp = dt.datetime.now(JST).strftime("%Y%m%d_%H%M%S")
    output_name = f"{timestamp}_{model_name}.csv"
    output_path = os.path.join(OUT_DIR, output_name)

    submission.to_csv(output_path, index=False, header=False)

    print("Saved:", output_path)
    print("rows :", len(submission))
    print(submission["10年定着確率"].describe().round(6))
    display(submission.head())
    return submission, output_path


In [5]:

# ============================================================
# ⑤ 前処理
# ============================================================
(
    persona_train,
    persona_test,
    monthly_train,
    monthly_test,
    monthly_train_full,
    sample_submission,
) = load_raw(PATHS)

audit_raw_data(
    persona_train,
    persona_test,
    monthly_train,
    monthly_test,
    monthly_train_full,
)

persona_train_use, monthly_train_use, early_leaver_ids = remove_early_leavers(
    persona_train,
    monthly_train,
)

print("\n=== 早期退職者除外 ===")
print("除外人数:", len(early_leaver_ids))
print("train人数:", len(persona_train), "->", len(persona_train_use))

assert monthly_train_use.groupby(ID_COLUMN).size().eq(24).all()
assert monthly_test.groupby(ID_COLUMN).size().eq(24).all()

train_feature = make_employee_features(
    persona_train_use,
    monthly_train_use,
)
test_feature = make_employee_features(
    persona_test,
    monthly_test,
)

assert train_feature[ID_COLUMN].is_unique
assert test_feature[ID_COLUMN].is_unique
assert set(test_feature[ID_COLUMN]) == set(sample_submission[ID_COLUMN])

X_all, y_all, X_test, cat_cols, feature_cols = prepare_catboost_matrix(
    train_feature,
    test_feature,
)

masks = make_time_masks(train_feature)

print("\n=== Feature / split ===")
print("train_feature:", train_feature.shape)
print("test_feature :", test_feature.shape)
print("n_features   :", len(feature_cols))
print("categorical  :", len(cat_cols))
print("prior cutoff :", masks["prior_cutoff"].date())
print("main cutoff  :", masks["main_cutoff"].date())
print("cal train/valid :", int(masks["cal_train"].sum()), int(masks["cal_valid"].sum()))
print("main train/valid:", int(masks["main_train"].sum()), int(masks["main_valid"].sum()))
print("main valid target mean:", round(float(y_all.loc[masks["main_valid"]].mean()), 5))


persona_train      : (2761, 20)
persona_test       : (2502, 19)
monthly_train      : (65754, 29) months 0 〜 23
monthly_test       : (60048, 29) months 0 〜 23
monthly_train_full : (257509, 29) months 0 〜 119
※ train_full の24〜119か月は特徴量に使用しません。

=== 早期退職者除外 ===
除外人数: 129
train人数: 2761 -> 2632

=== Feature / split ===
train_feature: (2632, 237)
test_feature : (2502, 236)
n_features   : 230
categorical  : 41
prior cutoff : 2012-04-01
main cutoff  : 2013-04-01
cal train/valid : 846 911
main train/valid: 1757 875
main valid target mean: 0.59657


In [6]:

# ============================================================
# ⑥ 推論
# ============================================================

PLAIN_SEEDS = [42, 777]
ORDERED_SEED = 42
BLEND_WEIGHT_PLAIN = 0.50

def make_plain_model(seed, iterations):
    return CatBoostClassifier(
        iterations=int(iterations),
        learning_rate=0.035,
        depth=7,
        loss_function="Logloss",
        eval_metric="Logloss",
        l2_leaf_reg=8.0,
        random_strength=0.7,
        random_seed=int(seed),
        verbose=False,
        allow_writing_files=False,
    )

def make_ordered_model(seed, iterations):
    return CatBoostClassifier(
        iterations=int(iterations),
        learning_rate=0.040,
        depth=6,
        boosting_type="Ordered",
        loss_function="Logloss",
        eval_metric="Logloss",
        l2_leaf_reg=10.0,
        random_strength=0.5,
        random_seed=int(seed),
        verbose=False,
        allow_writing_files=False,
    )

plain_cal_parts = []
for seed in PLAIN_SEEDS:
    m = make_plain_model(seed, 1100)
    m.fit(
        X_all.loc[masks["cal_train"]],
        y_all.loc[masks["cal_train"]],
        cat_features=cat_cols,
        eval_set=(
            X_all.loc[masks["cal_valid"]],
            y_all.loc[masks["cal_valid"]],
        ),
        early_stopping_rounds=90,
        verbose=False,
    )
    plain_cal_parts.append(
        m.predict_proba(
            X_all.loc[masks["cal_valid"]]
        )[:, 1]
    )

plain_cal_raw = np.mean(plain_cal_parts, axis=0)
plain_calibrator = fit_platt_calibrator(
    plain_cal_raw,
    y_all.loc[masks["cal_valid"]],
)

ordered_cal_model = make_ordered_model(
    ORDERED_SEED,
    600,
)
ordered_cal_model.fit(
    X_all.loc[masks["cal_train"]],
    y_all.loc[masks["cal_train"]],
    cat_features=cat_cols,
    eval_set=(
        X_all.loc[masks["cal_valid"]],
        y_all.loc[masks["cal_valid"]],
    ),
    early_stopping_rounds=60,
    verbose=False,
)
ordered_cal_raw = ordered_cal_model.predict_proba(
    X_all.loc[masks["cal_valid"]]
)[:, 1]

ordered_calibrator = fit_platt_calibrator(
    ordered_cal_raw,
    y_all.loc[masks["cal_valid"]],
)

print("\n=== Calibration fold ===")
evaluate_prob(
    y_all.loc[masks["cal_valid"]],
    plain_cal_raw,
    "Plain raw",
)
evaluate_prob(
    y_all.loc[masks["cal_valid"]],
    ordered_cal_raw,
    "Ordered raw",
)

plain_main_parts = []
plain_best_iterations = {}

for seed in PLAIN_SEEDS:
    m = make_plain_model(seed, 1200)
    m.fit(
        X_all.loc[masks["main_train"]],
        y_all.loc[masks["main_train"]],
        cat_features=cat_cols,
        eval_set=(
            X_all.loc[masks["main_valid"]],
            y_all.loc[masks["main_valid"]],
        ),
        early_stopping_rounds=90,
        verbose=False,
    )
    plain_best_iterations[seed] = max(
        50,
        int(m.get_best_iteration()) + 1,
    )
    plain_main_parts.append(
        m.predict_proba(
            X_all.loc[masks["main_valid"]]
        )[:, 1]
    )

plain_main_raw = np.mean(plain_main_parts, axis=0)
plain_main_cal = apply_platt(
    plain_calibrator,
    plain_main_raw,
)

ordered_main_model = make_ordered_model(
    ORDERED_SEED,
    700,
)
ordered_main_model.fit(
    X_all.loc[masks["main_train"]],
    y_all.loc[masks["main_train"]],
    cat_features=cat_cols,
    eval_set=(
        X_all.loc[masks["main_valid"]],
        y_all.loc[masks["main_valid"]],
    ),
    early_stopping_rounds=60,
    verbose=False,
)
ordered_best_iteration = max(
    50,
    int(ordered_main_model.get_best_iteration()) + 1,
)
ordered_main_raw = ordered_main_model.predict_proba(
    X_all.loc[masks["main_valid"]]
)[:, 1]
ordered_main_cal = apply_platt(
    ordered_calibrator,
    ordered_main_raw,
)

hybrid_valid = (
    BLEND_WEIGHT_PLAIN * plain_main_cal
    + (1.0 - BLEND_WEIGHT_PLAIN) * ordered_main_cal
)

print("\n=== Main Validation (2013-04〜2014-03) ===")
evaluate_prob(
    y_all.loc[masks["main_valid"]],
    plain_main_cal,
    "Plain calibrated",
)
evaluate_prob(
    y_all.loc[masks["main_valid"]],
    ordered_main_cal,
    "Ordered calibrated",
)
evaluate_prob(
    y_all.loc[masks["main_valid"]],
    hybrid_valid,
    "Hybrid 50:50",
)

plain_test_parts = []
for seed in PLAIN_SEEDS:
    m = make_plain_model(
        seed,
        plain_best_iterations[seed],
    )
    m.fit(
        X_all,
        y_all,
        cat_features=cat_cols,
        verbose=False,
    )
    plain_test_parts.append(
        m.predict_proba(X_test)[:, 1]
    )

plain_test_raw = np.mean(plain_test_parts, axis=0)
plain_test_cal = apply_platt(
    plain_calibrator,
    plain_test_raw,
)

ordered_final = make_ordered_model(
    ORDERED_SEED,
    ordered_best_iteration,
)
ordered_final.fit(
    X_all,
    y_all,
    cat_features=cat_cols,
    verbose=False,
)
ordered_test_raw = ordered_final.predict_proba(
    X_test
)[:, 1]
ordered_test_cal = apply_platt(
    ordered_calibrator,
    ordered_test_raw,
)

test_pred = (
    BLEND_WEIGHT_PLAIN * plain_test_cal
    + (1.0 - BLEND_WEIGHT_PLAIN) * ordered_test_cal
)



=== Calibration fold ===
Plain raw                            LogLoss=0.584214  AUC=0.723749  pred_mean=0.60930
Ordered raw                          LogLoss=0.576991  AUC=0.732466  pred_mean=0.60912

=== Main Validation (2013-04〜2014-03) ===
Plain calibrated                     LogLoss=0.547261  AUC=0.763158  pred_mean=0.60468
Ordered calibrated                   LogLoss=0.544602  AUC=0.766115  pred_mean=0.59549
Hybrid 50:50                         LogLoss=0.543435  AUC=0.768020  pred_mean=0.60009


In [7]:

# ============================================================
# ⑦ 提出ファイル出力
# ============================================================
submission, output_path = save_submission(
    sample_submission,
    test_feature,
    test_pred,
    MODEL_NAME,
)


Saved: /content/drive/MyDrive/Colab Notebooks/output/20260812_232846_Sample0812.csv
rows : 2502
count    2502.000000
mean        0.574436
std         0.260335
min         0.011256
25%         0.440950
50%         0.641399
75%         0.772706
max         0.976158
Name: 10年定着確率, dtype: float64


,社員ID,10年定着確率
0,E000002,0.581908
1,E000003,0.382900
2,E000004,0.763258
3,E000006,0.712258
4,E000009,0.550399
